In [ ]:
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Leemos el archivo generado por la aplicación de lecturas de wifis**

Convertimos las lineas que ponen la fecha de lectura en números de lectura que
será una nueva columna del csv resultante

In [ ]:
ruta='/content/drive/MyDrive/Archivos .csv/RegistroBSSIs.csv'
try:
    archivo = open(ruta, 'r')
    destino = open('/content/drive/MyDrive/Archivos .csv/RegistroWifisCentro.csv','w')
    destino.write('Lect,BSSid,Pot,Ubicacion\n')
    linea = archivo.readline()
    nlectura=0
    while linea:
        if ('Lectura' in linea):
            nlectura+=1
        else:
            destino.write(f'{nlectura},{linea}')
        linea = archivo.readline()
    archivo.close()
    destino.close()
except FileNotFoundError:
    print(f'El archivo {ruta} no existe.')

1.  `import pandas as pd`: **Importamos la librería pandas**.
2.  `lecturas=pd.read_csv(...)`: Leemos el archivo CSV  y lo guardamos en la variable `lecturas`.
3.  `lecturas.head()`: Mostramos las **primeras 5 filas** de la tabla `lecturas` para verificar que los datos se hayan cargado correctamente.

In [ ]:
import pandas as pd
lecturas=pd.read_csv('/content/drive/MyDrive/Archivos .csv/RegistroWifisCentro.csv')
lecturas.head()

,Lect,BSSid,Pot,Ubicacion
0,1,c0:51:5c:99:ca:64,-94,Administracion
1,1,ea:81:75:c6:b3:7b,-94,Administracion
2,1,56:59:19:bb:95:96,-85,Administracion
3,1,e2:41:36:c9:85:d0,-86,Administracion
4,1,3c:a7:ae:98:07:98,-81,Administracion


#**Obtenemos los BSSIDs con la cuenta de lecturas que tienen**

1.  `lecturas.groupby('BSSid')`: **Agrupamos** todas las filas de la tabla `lecturas` por el código único de cada punto de acceso (`BSSid`).
2.  `['Lect'].count()`: Una vez agrupados, cuenta cuántos registros hay para cada grupo, usando la columna `Lect` (que es el número de veces que se detectó ese BSSid).
3.  El resultado se guarda en la variable `lectA`. `lectA.head()` y mostramos los primeros BSSids y su **conteo total de apariciones**.

In [ ]:
lectA = lecturas.groupby('BSSid')['Lect'].count()
lectA.head()

,Lect
BSSid,
00:01:33:88:20:54,37
00:0f:66:e9:ab:ec,92
00:54:af:66:89:cb,1
00:54:af:81:58:ce,1
04:f0:21:63:18:b2,130


#**Filtramos los que tengan un número de lecturas menor o igual a 10**

1.  `lectA[lectA >= 10]`: De la lista de conteos (`lectA`), solo seleccionamos aquellos BSSids cuyo **conteo de apariciones es igual o mayor a 10**.
2.  Esto es crucial para **eliminar el "ruido"** (puntos de acceso que aparecen y desaparecen esporádicamente) y centrarse solo en las señales Wi-Fi más confiables para la localización.
3.  El resultado filtrado se guarda en la variable `lectB`. `lectB.head()` y mostramos los BSSids que sí pasaron el filtro.

In [ ]:
lectB = lectA[lectA >= 10]
lectB.head()

,Lect
BSSid,
00:01:33:88:20:54,37
00:0f:66:e9:ab:ec,92
04:f0:21:63:18:b2,130
26:39:4e:c8:ed:3c,59
26:60:b6:4d:5c:fc,54


#

#**Obtenemos la columna solo con los BSSId's**

1.  `lectB.index`: Extrae los códigos BSSid que forman el índice de la serie filtrada `lectB`.
2.  `pd.DataFrame({'BSSid': lectB.index})`: Crea una **nueva y pequeña tabla** (`lectC`) con una sola columna (`BSSid`) que contiene la lista final de los puntos de acceso que queremos mantener.
3.  Este paso es solo para tener la lista de BSSids en un formato fácil de usar (un DataFrame) para la siguiente operación de unión de tablas.

In [ ]:
# Opción 1
lectC = pd.DataFrame({'BSSid': lectB.index})

lectC.head()

# Opción 2

# lectC = lectB.index
# lectC_serie = pd.Series(lectC)
# lectC_serie.head()


,BSSid
0,00:01:33:88:20:54
1,00:0f:66:e9:ab:ec
2,04:f0:21:63:18:b2
3,26:39:4e:c8:ed:3c
4,26:60:b6:4d:5c:fc


#**Hacemos merge con el dataframe lecturas y la columna anterior, para que solo aparezcan los datos de puntos de acceso que tiene mas de 10 lecturas**
##**Podemos obtener también las etiquetas con groupby('Lect').sample(1), obtiene una sola fila de cada grupo**

1.  `lecturas.merge(lectC, on = 'BSSid', how = 'inner')`: Realiza una **unión interna** (`inner merge`) entre la tabla grande (`lecturas`) y la lista pequeña (`lectC`).
2.  **Unión Interna (`inner`):** Solo mantiene las filas de `lecturas` donde el valor de `BSSid` también existe en la lista `lectC`.
3.  El resultado es una **nueva tabla (`lecturas_new`)** que contiene la misma información original, pero **solo para los puntos de acceso Wi-Fi que consideramos importantes**.

In [ ]:
lecturas_new = lecturas.merge(lectC, on = 'BSSid', how = 'inner')
lecturas_new

,Lect,BSSid,Pot,Ubicacion
0,1,c0:51:5c:99:ca:64,-94,Administracion
1,1,ea:81:75:c6:b3:7b,-94,Administracion
2,1,56:59:19:bb:95:96,-85,Administracion
3,1,e2:41:36:c9:85:d0,-86,Administracion
4,1,3c:a7:ae:98:07:98,-81,Administracion
...,...,...,...,...
13461,651,44:12:44:87:5d:d1,-96,SalaVisitas
13462,651,94:bf:c4:a7:4d:ec,-80,SalaVisitas
13463,651,44:12:44:87:5d:d0,-94,SalaVisitas
13464,651,44:12:44:88:c9:a1,-73,SalaVisitas


#**Hacemos el pivotaje para que los BSSid's sean cabeceras de columna y sus datos sean simplemente las potencias**

1.  `lect_pivot = lecturas_new.pivot_table(...)`: **Crea una tabla pivote** :
    * `index = 'Lect'`: Cada fila será una lectura/medición única.
    * `columns = 'BSSid'`: Cada BSSid válido será una columna.
    * `values = 'Pot'`: La celda tendrá el valor de la potencia de la señal.
2.  `lect_pivot2 = lect_pivot.fillna(0)`: Los puntos donde un BSSid **no fue detectado** en una lectura específica quedan vacíos (NaN). Esta línea **rellena esos vacíos con `0.0`**. Esto es una convención común: `0.0` significa "no detectado" (o señal extremadamente débil).

In [ ]:
lect_pivot = lecturas_new.pivot_table(index = 'Lect', columns = 'BSSid', values = 'Pot')
lect_pivot2 = lect_pivot.fillna(0)
lect_pivot2

BSSid,00:01:33:88:20:54,00:0f:66:e9:ab:ec,04:f0:21:63:18:b2,26:39:4e:c8:ed:3c,26:60:b6:4d:5c:fc,28:77:77:dc:3d:68,28:77:77:dc:3d:a9,2a:77:77:cc:3d:a9,30:b5:c2:7f:37:94,30:b5:c2:7f:37:95,...,d4:86:60:e9:94:6b,e2:41:36:c9:85:d0,e4:3e:d7:a9:6a:bf,e4:3e:d7:a9:6a:c0,e8:48:b8:21:3f:b6,e8:48:b8:21:3f:b7,ea:63:7b:a4:2f:6e,ea:81:75:c6:b3:7b,ee:68:68:81:71:29,fc:12:63:c2:13:1f
Lect,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-85.0,-67.0,...,0.0,-86.0,0.0,0.0,0.0,0.0,0.0,-94.0,-73.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-85.0,-67.0,...,0.0,-86.0,0.0,0.0,0.0,0.0,0.0,-94.0,-73.0,0.0
3,0.0,0.0,0.0,-91.0,0.0,0.0,0.0,0.0,-92.0,-67.0,...,0.0,0.0,0.0,0.0,0.0,-82.0,0.0,0.0,-81.0,0.0
4,0.0,0.0,0.0,-91.0,0.0,0.0,0.0,0.0,-92.0,-67.0,...,0.0,0.0,0.0,0.0,0.0,-82.0,0.0,0.0,-81.0,0.0
5,0.0,0.0,0.0,-91.0,0.0,0.0,0.0,0.0,-92.0,-67.0,...,0.0,0.0,0.0,0.0,0.0,-82.0,0.0,0.0,-81.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-83.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-91.0,0.0,0.0,0.0
648,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-84.0,-68.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-94.0,0.0,0.0,0.0
649,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-84.0,-68.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#**Le volvemos a pegar la columna ubicación si queremos ver los datos de las potencias de las wifis junto a su ubicación.**

1.  `mapa_ubicacion = lecturas[['Lect', 'Ubicacion']].drop_duplicates().set_index('Lect')`: Crea una pequeña tabla de ayuda para asegurar que **cada número de lectura (`Lect`) tenga una única ubicación** asignada, y usa `Lect` como índice.
2.  `tabla_pivote_final = lect_pivot2.join(mapa_ubicacion)`: **Une (join)** la matriz de características (`lect_pivot2`) con la columna de ubicaciones (`mapa_ubicacion`), usando el número de lectura (`Lect`) como punto de conexión.
3.  El resultado (`tabla_pivote_final`) es el **Dataset final** listo para el análisis, con **651 filas** (lecturas) y **111 columnas** (110 BSSids y 1 Ubicación).

In [ ]:
# Extraemos la columna Ubicacion asegurándonos de que solo haya una Ubicacion por Lectura (eliminando duplicados)
# y la establecemos como un DataFrame con el índice Lect.
mapa_ubicacion = lecturas[['Lect', 'Ubicacion']].drop_duplicates().set_index('Lect')

# UNIR la columna Ubicacion al resultado pivote
tabla_pivote_final = lect_pivot2.join(mapa_ubicacion)

tabla_pivote_final

,00:01:33:88:20:54,00:0f:66:e9:ab:ec,04:f0:21:63:18:b2,26:39:4e:c8:ed:3c,26:60:b6:4d:5c:fc,28:77:77:dc:3d:68,28:77:77:dc:3d:a9,2a:77:77:cc:3d:a9,30:b5:c2:7f:37:94,30:b5:c2:7f:37:95,...,e2:41:36:c9:85:d0,e4:3e:d7:a9:6a:bf,e4:3e:d7:a9:6a:c0,e8:48:b8:21:3f:b6,e8:48:b8:21:3f:b7,ea:63:7b:a4:2f:6e,ea:81:75:c6:b3:7b,ee:68:68:81:71:29,fc:12:63:c2:13:1f,Ubicacion
Lect,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-85.0,-67.0,...,-86.0,0.0,0.0,0.0,0.0,0.0,-94.0,-73.0,0.0,Administracion
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-85.0,-67.0,...,-86.0,0.0,0.0,0.0,0.0,0.0,-94.0,-73.0,0.0,Administracion
3,0.0,0.0,0.0,-91.0,0.0,0.0,0.0,0.0,-92.0,-67.0,...,0.0,0.0,0.0,0.0,-82.0,0.0,0.0,-81.0,0.0,Administracion
4,0.0,0.0,0.0,-91.0,0.0,0.0,0.0,0.0,-92.0,-67.0,...,0.0,0.0,0.0,0.0,-82.0,0.0,0.0,-81.0,0.0,Administracion
5,0.0,0.0,0.0,-91.0,0.0,0.0,0.0,0.0,-92.0,-67.0,...,0.0,0.0,0.0,0.0,-82.0,0.0,0.0,-81.0,0.0,Administracion
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-83.0,0.0,...,0.0,0.0,0.0,0.0,0.0,-91.0,0.0,0.0,0.0,SalaVisitas
648,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-84.0,-68.0,...,0.0,0.0,0.0,0.0,0.0,-94.0,0.0,0.0,0.0,SalaVisitas
649,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-84.0,-68.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SalaVisitas


1.  `tabla_pivote_final.to_csv('datasetwifis-3.csv')`: Toma la tabla final (el dataset limpio y pivoteado) y la guarda como un archivo llamado `datasetwifis-3.csv` en la carpeta actual de trabajo.

In [ ]:
#Guardamos el dataframe dataset como datasetwifis.csv
tabla_pivote_final.to_csv('datasetwifis-3.csv')